# 02 Cleaning

This notebook performs all data quality steps on the raw Swiggy dataset:
1. Drop null rows
2. Remove duplicate rows
3. Filter out Rating Count = 0 rows
4. Flag price outliers using the IQR method
5. Final Load Prep: Rename columns, create derived features (month, day_of_week, price_bucket), and cast data types.

In [9]:
from pathlib import Path
import pandas as pd

current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir.parent if current_dir.name.strip() == 'notebooks' else current_dir

In [10]:
RAW_PATH = PROJECT_ROOT / 'data/raw/swiggy_data_raw.csv'
df = pd.read_csv(RAW_PATH, parse_dates=['Order Date'])
print('Loaded:', df.shape)
df.head()

Loaded: (197430, 10)


,State,City,Order Date,Restaurant Name,Location,Category,Dish Name,Price (INR),Rating,Rating Count
0,Karnataka,Bengaluru,2025-06-29,Anand Sweets & Savouries,Rajarajeshwari Nagar,Snack,Butter Murukku-200gm,133.9,4.0,0.0
1,Karnataka,Bengaluru,2025-04-03,Srinidhi Sagar Deluxe,Kengeri,Recommended,Badam Milk,52.0,4.5,25.0
2,Karnataka,Bengaluru,2025-01-15,Srinidhi Sagar Deluxe,Kengeri,Recommended,Chow Chow Bath,117.0,4.7,48.0
3,Karnataka,Bengaluru,2025-04-17,Srinidhi Sagar Deluxe,Kengeri,Recommended,Kesari Bath,65.0,4.6,NaN
4,Karnataka,Bengaluru,2025-03-13,Srinidhi Sagar Deluxe,Kengeri,Recommended,Mix Raitha,130.0,4.0,0.0


## Step 1 — Drop Null Rows

In [11]:
before = len(df)
df = df.dropna()
after = len(df)
print(f'Before: {before:,} rows')
print(f'After dropping nulls: {after:,} rows  (-{before - after:,})')

Before: 197,430 rows
After dropping nulls: 152,894 rows  (-44,536)


## Step 2 — Remove Duplicate Rows

In [12]:
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f'After removing duplicates: {after:,} rows  (-{before - after:,})')

After removing duplicates: 152,882 rows  (-12)


## Step 3 — Filter Out Rating Count = 0

In [13]:
before = len(df)
df = df[df['Rating Count'] != 0]
after = len(df)
print(f'After filtering Rating Count = 0: {after:,} rows  (-{before - after:,})')

After filtering Rating Count = 0: 91,788 rows  (-61,094)


## Step 4 — Flag Price Outliers (IQR Method)

In [14]:
Q1 = df['Price (INR)'].quantile(0.25)
Q3 = df['Price (INR)'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df['is_price_outlier'] = ~df['Price (INR)'].between(lower_bound, upper_bound)

print(f'IQR Bounds  →  Lower: {lower_bound:.2f}  |  Upper: {upper_bound:.2f}')
print(f'Price outliers flagged: {df["is_price_outlier"].sum():,} rows')

IQR Bounds  →  Lower: -130.50  |  Upper: 577.50
Price outliers flagged: 4,180 rows


## Step 5 — Final Load Prep

To align with the Data Dictionary, we will:
1. Rename columns to `snake_case`.
2. Create derived columns: `month`, `day_of_week`, and `price_bucket`.
3. Convert `rating_count` to integer.
4. Remove the `is_price_outlier` flag (used only for internal filtering).

In [16]:
column_map = {
    'State': 'state', 
    'City': 'city', 
    'Order Date': 'order_date', 
    'Restaurant Name': 'restaurant_name', 
    'Location': 'location', 
    'Category': 'category', 
    'Dish Name': 'dish_name', 
    'Price (INR)': 'price_inr', 
    'Rating': 'rating', 
    'Rating Count': 'rating_count'
}
df = df.rename(columns=column_map)

df['month'] = df['order_date'].dt.to_period('M').astype(str)
df['day_of_week'] = df['order_date'].dt.day_name()

def categorize_price(price):
    if price < 150:
        return 'Budget'
    elif price < 400:
        return 'Mid-range'
    else:
        return 'Premium'

df['price_bucket'] = df['price_inr'].apply(categorize_price)

df['rating_count'] = df['rating_count'].astype(int)

# is_price_outlier is kept for EDA purposes

print('Final Columns:', df.columns.tolist())
df.head()

Final Columns: ['state', 'city', 'order_date', 'restaurant_name', 'location', 'category', 'dish_name', 'price_inr', 'rating', 'rating_count', 'is_price_outlier', 'month', 'day_of_week', 'price_bucket']


,state,city,order_date,restaurant_name,location,category,dish_name,price_inr,rating,rating_count,is_price_outlier,month,day_of_week,price_bucket
1,Karnataka,Bengaluru,2025-04-03,Srinidhi Sagar Deluxe,Kengeri,Recommended,Badam Milk,52.0,4.5,25,False,2025-04,Thursday,Budget
2,Karnataka,Bengaluru,2025-01-15,Srinidhi Sagar Deluxe,Kengeri,Recommended,Chow Chow Bath,117.0,4.7,48,False,2025-01,Wednesday,Budget
6,Karnataka,Bengaluru,2025-01-21,Srinidhi Sagar Deluxe,Kengeri,Recommended,Garlic Naan,98.0,4.0,34,False,2025-01,Tuesday,Budget
8,Karnataka,Bengaluru,2025-05-02,Srinidhi Sagar Deluxe,Kengeri,North Indian Gravy,Panneer Butter Masala,241.0,4.4,29,False,2025-05,Friday,Mid-range
9,Karnataka,Bengaluru,2025-07-30,Srinidhi Sagar Deluxe,Kengeri,North Indian Gravy,Dal Tadka,195.0,4.9,51,False,2025-07,Wednesday,Mid-range


## Save Final Cleaned Data

In [17]:
PROCESSED_PATH = PROJECT_ROOT / 'data/processed/swiggy_cleaned.csv'
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_PATH, index=False)
print(f'Final cleaned data saved to: {PROCESSED_PATH}')

Final cleaned data saved to: /Users/kshitijsaxena/Desktop/Hopper_C-2_SwiggyDataAnalysis/data/processed/swiggy_cleaned.csv
